# HMM Emission State Analysis
### NVIDIA Developer Lifecycle Project

Analyzes the distribution and transition behavior of weekly GMM emission states
(used as observations for HMM modeling) across developer lifecycle strata and adoption directions.

| State | Label | Description |
|---|---|---|
| c0 | Moderate | Regular engagement weeks |
| c1 | Burst | High-intensity engagement weeks (rarest, highest confidence) |
| c2 | Inactive | Low / no activity weeks |

**Sections**
1. HMM State Characterization
2. State Distribution by Lifecycle Stratum
3. State Distribution by Adoption Direction
4. Transition Matrices by Stratum
5. Developer-Level State Metrics by Stratum
6. Temporal Trends by Stratum
7. Burst-State Developer Profile
8. Summary Table

## Setup — Imports, Config & Helpers

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
PARQUET_DIR = Path(r"C:\Users\wbdor\Downloads\toexport_clusters")
OUTPUT_DIR  = Path(r"C:\Users\wbdor\OneDrive\Public\Important resources\Codex & Claude Code\hmm_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = OUTPUT_DIR / "hmm_demographic_analysis.xlsx"
PDF_PATH   = OUTPUT_DIR / "hmm_demographic_analysis.pdf"

# ── Style ─────────────────────────────────────────────────────────────────────
NVIDIA_GREEN  = "#76B900"
STATE_LABELS  = {0: "Moderate (c0)", 1: "Burst (c1)", 2: "Inactive (c2)"}
STATE_COLORS  = {0: NVIDIA_GREEN, 1: "#1F77B4", 2: "#D62728"}
STRATUM_ORDER = ["active", "cooling", "at_risk", "dormant", "unactivated"]
ADOPTION_ORDER = [
    "accelerating_or_active", "declining", "at_risk",
    "steady_inactive", "not_activated",
]

# ── State ─────────────────────────────────────────────────────────────────────
con      = duckdb.connect()
_sheets  = {}   # collected DataFrames, written to Excel at the end
_pdf     = None # PdfPages handle, opened on first plot

# ── Helpers ───────────────────────────────────────────────────────────────────
def q(sql):
    return con.execute(sql).df()

def keep(df, sheet):
    _sheets[sheet] = df

def save(fig, title):
    global _pdf
    if _pdf is None:
        _pdf = PdfPages(PDF_PATH)
    _pdf.savefig(fig, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def pct_fmt(x, _):
    return f"{x:.0f}%"

## Load Data

In [ ]:
weekly_path = str(PARQUET_DIR / "dev_gmm_weekly_clusters_v1.parquet")
final_path  = str(PARQUET_DIR / "dev_lifecycle_cluster_membership_v11_final.parquet")

con.execute(f"CREATE OR REPLACE VIEW weekly   AS SELECT * FROM read_parquet('{weekly_path}')")
con.execute(f"CREATE OR REPLACE VIEW lc_final AS SELECT * FROM read_parquet('{final_path}')")

print("Views registered: weekly, lc_final")

## 1. HMM State Characterization
Observation and developer counts for each weekly emission state, plus average assignment confidence.

In [ ]:
state_char = q("""
    SELECT
        gmm_weekly_cluster_id                         AS state,
        COUNT(*)                                      AS n_observations,
        COUNT(DISTINCT developer_id)                  AS n_developers,
        ROUND(AVG(gmm_weekly_max_posterior), 4)       AS avg_confidence,
        ROUND(STDDEV(gmm_weekly_max_posterior), 4)    AS std_confidence,
        ROUND(AVG(gmm_weekly_prob_c0), 4)             AS avg_prob_c0,
        ROUND(AVG(gmm_weekly_prob_c1), 4)             AS avg_prob_c1,
        ROUND(AVG(gmm_weekly_prob_c2), 4)             AS avg_prob_c2
    FROM weekly
    GROUP BY gmm_weekly_cluster_id
    ORDER BY gmm_weekly_cluster_id
""")
state_char["state_label"] = state_char["state"].map(STATE_LABELS)
keep(state_char, "1_state_characterization")
state_char

In [ ]:
colors = [STATE_COLORS[s] for s in state_char["state"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Weekly GMM State Characterization", fontsize=14, fontweight="bold")

axes[0].bar(state_char["state_label"], state_char["n_observations"] / 1e6, color=colors)
axes[0].set_title("Observations per State (M)")
axes[0].set_ylabel("Millions")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(state_char["state_label"], state_char["n_developers"] / 1e6, color=colors)
axes[1].set_title("Unique Developers per State (M)")
axes[1].set_ylabel("Millions")
axes[1].tick_params(axis="x", rotation=15)

for ax in axes:
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}M", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
save(fig, "1 — State Characterization")

## 2. State Distribution by Lifecycle Stratum
What share of weekly observations fall into each emission state, broken down by lifecycle stratum.

In [ ]:
state_by_stratum = q("""
    SELECT
        f.stratum,
        w.gmm_weekly_cluster_id AS state,
        COUNT(*)                AS n_observations,
        COUNT(DISTINCT w.developer_id) AS n_developers
    FROM weekly w
    JOIN lc_final f ON w.developer_id = f.developer_id
    GROUP BY f.stratum, w.gmm_weekly_cluster_id
    ORDER BY f.stratum, w.gmm_weekly_cluster_id
""")
state_by_stratum["pct_obs"] = (
    state_by_stratum.groupby("stratum")["n_observations"]
    .transform(lambda x: x / x.sum() * 100)
)
state_by_stratum["state_label"] = state_by_stratum["state"].map(STATE_LABELS)
keep(state_by_stratum, "2_state_by_stratum")
state_by_stratum

In [ ]:
pivot = (
    state_by_stratum
    .pivot(index="stratum", columns="state_label", values="pct_obs")
    .reindex(STRATUM_ORDER).fillna(0)
)

fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(pivot))
for col, color in zip(pivot.columns, [STATE_COLORS[k] for k in sorted(STATE_COLORS)]):
    bars = ax.bar(pivot.index, pivot[col], bottom=bottom, label=col, color=color)
    for bar, b in zip(bars, bottom):
        h = bar.get_height()
        if h > 3:
            ax.text(bar.get_x() + bar.get_width() / 2, b + h / 2,
                    f"{h:.1f}%", ha="center", va="center", fontsize=8,
                    color="white", fontweight="bold")
    bottom += pivot[col].values

ax.set_title("Weekly HMM State Distribution by Lifecycle Stratum", fontsize=13, fontweight="bold")
ax.set_ylabel("% of Weekly Observations")
ax.set_xlabel("Lifecycle Stratum")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
ax.legend(title="Weekly State", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
save(fig, "2 — State Distribution by Stratum")

## 3. State Distribution by Adoption Direction
Same breakdown using adoption direction labels (accelerating, declining, at-risk, steady inactive, not activated).

In [ ]:
state_by_adoption = q("""
    SELECT
        f.adoption_direction,
        w.gmm_weekly_cluster_id AS state,
        COUNT(*)                AS n_observations,
        COUNT(DISTINCT w.developer_id) AS n_developers
    FROM weekly w
    JOIN lc_final f ON w.developer_id = f.developer_id
    GROUP BY f.adoption_direction, w.gmm_weekly_cluster_id
    ORDER BY f.adoption_direction, w.gmm_weekly_cluster_id
""")
state_by_adoption["pct_obs"] = (
    state_by_adoption.groupby("adoption_direction")["n_observations"]
    .transform(lambda x: x / x.sum() * 100)
)
state_by_adoption["state_label"] = state_by_adoption["state"].map(STATE_LABELS)
keep(state_by_adoption, "3_state_by_adoption")
state_by_adoption

In [ ]:
pivot_ad = (
    state_by_adoption
    .pivot(index="adoption_direction", columns="state_label", values="pct_obs")
    .reindex(ADOPTION_ORDER).fillna(0)
)

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(pivot_ad))
for col, color in zip(pivot_ad.columns, [STATE_COLORS[k] for k in sorted(STATE_COLORS)]):
    bars = ax.bar(pivot_ad.index, pivot_ad[col], bottom=bottom, label=col, color=color)
    for bar, b in zip(bars, bottom):
        h = bar.get_height()
        if h > 3:
            ax.text(bar.get_x() + bar.get_width() / 2, b + h / 2,
                    f"{h:.1f}%", ha="center", va="center", fontsize=8,
                    color="white", fontweight="bold")
    bottom += pivot_ad[col].values

ax.set_title("Weekly HMM State Distribution by Adoption Direction", fontsize=13, fontweight="bold")
ax.set_ylabel("% of Weekly Observations")
ax.set_xlabel("Adoption Direction")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
ax.legend(title="Weekly State", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
save(fig, "3 — State Distribution by Adoption Direction")

## 4. Transition Matrices by Lifecycle Stratum
Week-over-week state transition probabilities for each stratum. Only consecutive weekly observations (7-day gap) are counted.

In [ ]:
transitions_raw = q("""
    WITH ordered AS (
        SELECT
            w.developer_id,
            f.stratum,
            w.week_start,
            w.gmm_weekly_cluster_id AS state,
            LEAD(w.gmm_weekly_cluster_id) OVER (
                PARTITION BY w.developer_id ORDER BY w.week_start
            ) AS next_state,
            LEAD(w.week_start) OVER (
                PARTITION BY w.developer_id ORDER BY w.week_start
            ) AS next_week
        FROM weekly w
        JOIN lc_final f ON w.developer_id = f.developer_id
    )
    SELECT
        stratum,
        state       AS from_state,
        next_state  AS to_state,
        COUNT(*)    AS n
    FROM ordered
    WHERE next_state IS NOT NULL
      AND next_week = week_start + INTERVAL 7 DAYS
    GROUP BY stratum, state, next_state
    ORDER BY stratum, state, next_state
""")
keep(transitions_raw, "4_transitions_raw")
transitions_raw

In [ ]:
strata_in_transitions = [s for s in STRATUM_ORDER if s in transitions_raw["stratum"].unique()]
n_strata = len(strata_in_transitions)

fig, axes = plt.subplots(1, n_strata, figsize=(4 * n_strata, 4))
fig.suptitle("Week-over-Week HMM State Transition Probabilities by Lifecycle Stratum",
             fontsize=13, fontweight="bold")

for ax, stratum in zip(axes, strata_in_transitions):
    sub = transitions_raw[transitions_raw["stratum"] == stratum].copy()
    matrix = pd.DataFrame(0.0, index=range(3), columns=range(3))
    for _, row in sub.iterrows():
        matrix.loc[int(row["from_state"]), int(row["to_state"])] = row["n"]
    matrix = matrix.div(matrix.sum(axis=1), axis=0).fillna(0)
    matrix.index   = [STATE_LABELS[i] for i in range(3)]
    matrix.columns = [STATE_LABELS[i] for i in range(3)]

    sns.heatmap(matrix, ax=ax, annot=True, fmt=".2f", cmap="Greens",
                vmin=0, vmax=1, cbar=False, linewidths=0.5, linecolor="white")
    ax.set_title(stratum.capitalize(), fontsize=11, fontweight="bold")
    ax.set_xlabel("To State")
    ax.set_ylabel("From State" if ax == axes[0] else "")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
save(fig, "4 — Transition Matrices by Stratum")

## 5. Developer-Level State Metrics by Stratum
Per-developer averages: time in each state, Shannon entropy of state sequence, assignment confidence.

In [ ]:
dev_metrics = q("""
    WITH dev_state_counts AS (
        SELECT
            w.developer_id,
            f.stratum,
            f.adoption_direction,
            COUNT(*)                                    AS total_weeks,
            SUM(CASE WHEN w.gmm_weekly_cluster_id = 0 THEN 1 ELSE 0 END) AS weeks_moderate,
            SUM(CASE WHEN w.gmm_weekly_cluster_id = 1 THEN 1 ELSE 0 END) AS weeks_burst,
            SUM(CASE WHEN w.gmm_weekly_cluster_id = 2 THEN 1 ELSE 0 END) AS weeks_inactive,
            AVG(w.gmm_weekly_max_posterior)             AS avg_confidence
        FROM weekly w
        JOIN lc_final f ON w.developer_id = f.developer_id
        GROUP BY w.developer_id, f.stratum, f.adoption_direction
    ),
    with_entropy AS (
        SELECT *,
            weeks_moderate::DOUBLE / total_weeks AS pct_moderate,
            weeks_burst::DOUBLE    / total_weeks AS pct_burst,
            weeks_inactive::DOUBLE / total_weeks AS pct_inactive,
            -( CASE WHEN weeks_moderate > 0
                    THEN (weeks_moderate::DOUBLE/total_weeks) * LN(weeks_moderate::DOUBLE/total_weeks)
                    ELSE 0 END
             + CASE WHEN weeks_burst > 0
                    THEN (weeks_burst::DOUBLE/total_weeks)    * LN(weeks_burst::DOUBLE/total_weeks)
                    ELSE 0 END
             + CASE WHEN weeks_inactive > 0
                    THEN (weeks_inactive::DOUBLE/total_weeks) * LN(weeks_inactive::DOUBLE/total_weeks)
                    ELSE 0 END
            ) AS state_entropy
        FROM dev_state_counts
    )
    SELECT
        stratum, adoption_direction,
        COUNT(*)                                        AS n_developers,
        ROUND(AVG(total_weeks), 1)                     AS avg_weeks_observed,
        ROUND(MEDIAN(total_weeks), 0)                  AS median_weeks_observed,
        ROUND(AVG(pct_moderate)  * 100, 2)             AS avg_pct_moderate,
        ROUND(AVG(pct_burst)     * 100, 2)             AS avg_pct_burst,
        ROUND(AVG(pct_inactive)  * 100, 2)             AS avg_pct_inactive,
        ROUND(AVG(state_entropy), 4)                   AS avg_state_entropy,
        ROUND(AVG(avg_confidence), 4)                  AS avg_assignment_confidence
    FROM with_entropy
    GROUP BY stratum, adoption_direction
    ORDER BY stratum, adoption_direction
""")
keep(dev_metrics, "5_developer_metrics")
dev_metrics

In [ ]:
stratum_agg = dev_metrics.groupby("stratum")[
    ["avg_pct_moderate", "avg_pct_burst", "avg_pct_inactive"]
].mean().reindex(STRATUM_ORDER).dropna()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(stratum_agg))
width = 0.28
ax.bar(x - width, stratum_agg["avg_pct_moderate"], width, label=STATE_LABELS[0], color=STATE_COLORS[0])
ax.bar(x,         stratum_agg["avg_pct_burst"],    width, label=STATE_LABELS[1], color=STATE_COLORS[1])
ax.bar(x + width, stratum_agg["avg_pct_inactive"], width, label=STATE_LABELS[2], color=STATE_COLORS[2])
ax.set_xticks(x)
ax.set_xticklabels(stratum_agg.index, rotation=15)
ax.set_title("Average % of Weeks per Developer in Each State by Stratum", fontsize=13, fontweight="bold")
ax.set_ylabel("Avg % of Observed Weeks")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
ax.legend(title="Weekly State")
plt.tight_layout()
save(fig, "5a — Avg State Time by Stratum")

In [ ]:
stratum_entropy = dev_metrics.groupby("stratum")["avg_state_entropy"].mean().reindex(STRATUM_ORDER).dropna()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(stratum_entropy.index, stratum_entropy.values, color=NVIDIA_GREEN)
ax.set_title("Average Developer State Entropy by Lifecycle Stratum\n(Higher = more mixed weekly behavior)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Avg Shannon Entropy")
ax.tick_params(axis="x", rotation=15)
for i, v in enumerate(stratum_entropy.values):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
save(fig, "5b — State Entropy by Stratum")

## 6. Temporal Trends by Stratum
Quarterly state composition from 2022 to 2026, stacked area chart per stratum.

In [ ]:
temporal = q("""
    SELECT
        DATE_TRUNC('quarter', w.week_start) AS quarter,
        f.stratum,
        w.gmm_weekly_cluster_id             AS state,
        COUNT(*)                            AS n_observations
    FROM weekly w
    JOIN lc_final f ON w.developer_id = f.developer_id
    WHERE w.week_start >= '2022-01-01'
    GROUP BY DATE_TRUNC('quarter', w.week_start), f.stratum, w.gmm_weekly_cluster_id
    ORDER BY quarter, f.stratum, state
""")
temporal["quarter"] = pd.to_datetime(temporal["quarter"])
temporal["pct"] = (
    temporal.groupby(["quarter", "stratum"])["n_observations"]
    .transform(lambda x: x / x.sum() * 100)
)
temporal["state_label"] = temporal["state"].map(STATE_LABELS)
keep(temporal, "6_temporal_trends")
temporal.head()

In [ ]:
strata_to_plot = [s for s in STRATUM_ORDER if s in temporal["stratum"].unique()]
fig, axes = plt.subplots(len(strata_to_plot), 1, figsize=(14, 4 * len(strata_to_plot)), sharex=True)
fig.suptitle("Quarterly HMM State Composition Over Time by Stratum (2022–2026)",
             fontsize=13, fontweight="bold")

for ax, stratum in zip(axes, strata_to_plot):
    sub = temporal[temporal["stratum"] == stratum]
    pivot_t = sub.pivot_table(index="quarter", columns="state_label", values="pct", aggfunc="sum").fillna(0)
    ordered_cols = [STATE_LABELS[k] for k in sorted(STATE_COLORS) if STATE_LABELS[k] in pivot_t.columns]
    pivot_t[ordered_cols].plot.area(ax=ax, color=[STATE_COLORS[k] for k in sorted(STATE_COLORS)],
                                    alpha=0.85, legend=False)
    ax.set_title(f"Stratum: {stratum}", fontsize=10, fontweight="bold")
    ax.set_ylabel("% Observations")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
    ax.set_ylim(0, 100)

handles = [plt.Rectangle((0, 0), 1, 1, color=STATE_COLORS[k]) for k in sorted(STATE_COLORS)]
labels  = [STATE_LABELS[k] for k in sorted(STATE_COLORS)]
fig.legend(handles, labels, title="Weekly State", loc="upper right", bbox_to_anchor=(1.0, 0.98))
axes[-1].set_xlabel("Quarter")
plt.tight_layout()
save(fig, "6 — Temporal Trends by Stratum")

## 7. Burst-State Developer Profile
Which developers have ever been observed in the Burst (c1) state, segmented by stratum and adoption direction.

In [ ]:
burst_profile = q("""
    WITH burst_devs AS (
        SELECT DISTINCT developer_id
        FROM weekly
        WHERE gmm_weekly_cluster_id = 1
    )
    SELECT
        f.stratum,
        f.adoption_direction,
        COUNT(*) AS n_burst_developers
    FROM burst_devs b
    JOIN lc_final f ON b.developer_id = f.developer_id
    GROUP BY f.stratum, f.adoption_direction
    ORDER BY n_burst_developers DESC
""")
keep(burst_profile, "7_burst_state_profile")
burst_profile

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels_bp = burst_profile["stratum"] + "\n(" + burst_profile["adoption_direction"] + ")"
ax.barh(labels_bp[::-1], burst_profile["n_burst_developers"][::-1], color=STATE_COLORS[1])
ax.set_title("Developers Who Reached Burst State (c1) — by Stratum & Adoption Direction",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Number of Developers")
for i, v in enumerate(burst_profile["n_burst_developers"][::-1]):
    ax.text(v + 200, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
save(fig, "7 — Burst State Developer Profile")

## 8. Summary Table
High-level per-stratum summary: developer counts, total weeks observed, % time in each state, average assignment confidence.

In [ ]:
summary = q("""
    WITH base AS (
        SELECT
            f.stratum,
            f.adoption_direction,
            COUNT(DISTINCT w.developer_id)               AS n_developers,
            COUNT(*)                                     AS total_weeks,
            ROUND(AVG(CASE WHEN w.gmm_weekly_cluster_id = 0 THEN 1.0 ELSE 0.0 END) * 100, 1) AS pct_moderate_weeks,
            ROUND(AVG(CASE WHEN w.gmm_weekly_cluster_id = 1 THEN 1.0 ELSE 0.0 END) * 100, 1) AS pct_burst_weeks,
            ROUND(AVG(CASE WHEN w.gmm_weekly_cluster_id = 2 THEN 1.0 ELSE 0.0 END) * 100, 1) AS pct_inactive_weeks,
            ROUND(AVG(w.gmm_weekly_max_posterior), 4)    AS avg_assignment_confidence
        FROM weekly w
        JOIN lc_final f ON w.developer_id = f.developer_id
        GROUP BY f.stratum, f.adoption_direction
    )
    SELECT * FROM base ORDER BY stratum, adoption_direction
""")
keep(summary, "8_summary")
summary

## Save Outputs
Flush all collected DataFrames to a single Excel workbook and close the PDF.

In [ ]:
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    for sheet_name, df in _sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

if _pdf is not None:
    _pdf.close()

print(f"Excel: {EXCEL_PATH}")
print(f"PDF:   {PDF_PATH}")